# BanglaSlumNet × LocateAnything — Master Colab Notebook
**RESEARCH-ONLY.** See THIRD_PARTY_LICENSES.md.

## Phases
- Phase 0: Setup (Drive, repo, deps, GPU check)
- Phase 1: Models & data (one-time, cached)
- Phase 2: Weak labels (one-time, cached)
- Phase 3: Feature & SAS-Net caching (one-time)
- Phase 4: Experiments (cheap, resumable)
- Phase 5: Figures & tables

**Every heavy phase checks the cache/registry and skips if already done.**
Re-running after a disconnect resumes safely from where you left off.

## Phase 0 — Setup

In [ ]:
# Cell P0.1: Mount Drive and set PROJECT_ROOT
import sys, os, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/gdrive')
    PROJECT_ROOT = Path('/gdrive/MyDrive/BanglaSlumNet')
else:
    PROJECT_ROOT = Path('.').resolve().parent  # adjust if running locally

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
print(f'PROJECT_ROOT: {PROJECT_ROOT}')

In [ ]:
# Cell P0.2: Clone or pull repo
REPO_DIR = PROJECT_ROOT / 'repo'
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/namaray/BanglaSlumNet.git', str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)
os.chdir(str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR))
print(f'Repo at: {REPO_DIR}')

In [ ]:
# Cell P0.3: Install requirements (Colab-specific order)
# DO NOT install MagiAttention — not available on Ampere (A100/Colab)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements_colab.txt'], check=True)
print('Requirements installed.')

In [ ]:
# Cell P0.4: GPU check
import torch
subprocess.run(['nvidia-smi'], check=False)
if torch.cuda.is_available():
    gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {torch.cuda.get_device_name(0)} | {gb:.1f} GB')
    if gb < 24:
        print('WARNING: < 24 GB VRAM. Enable load_in_4bit in config or use T4 fallback.')
    DEVICE = 'cuda'
else:
    print('No GPU. Set runtime to GPU in Colab.')
    DEVICE = 'cpu'

In [ ]:
# Cell P0.5: Load config
from omegaconf import OmegaConf
cfg = OmegaConf.load('config/default.yaml')

# ── Decision flags (§15) — edit here, not in source code ─────────────────
cfg.decide = OmegaConf.create({
    'D1_feature_mode': 'hidden_state',   # hidden_state | grounding_map
    'D2_lora_adapt': False,
    'D3_atm_correction': 'classical',
    'D4_generalization': 'loro',
    'D6_include_gram': True,
    'D7_use_swir': False,
})

# Override paths to use Drive
cfg.paths.project_root = str(PROJECT_ROOT)
cfg.paths.data_dir = str(PROJECT_ROOT / 'data')
cfg.paths.tiles_dir = str(PROJECT_ROOT / 'data/tiles')
cfg.paths.labels_dir = str(PROJECT_ROOT / 'data/labels')
cfg.paths.socioeconomic_dir = str(PROJECT_ROOT / 'data/socioeconomic')
cfg.paths.features_cache_dir = str(PROJECT_ROOT / 'data/features_cache')
cfg.paths.results_dir = str(PROJECT_ROOT / 'results')
cfg.locate_anything.feature_mode = cfg.decide.D1_feature_mode

print(OmegaConf.to_yaml(cfg))

## Phase 1 — Models & Data (one-time, cached)

In [ ]:
# Cell P1.1: Download LocateAnything to Drive (run once; idempotent)
MODEL_CACHE = str(PROJECT_ROOT / 'model_cache')
SENTINEL = PROJECT_ROOT / 'model_cache' / 'download_done.flag'
if not SENTINEL.exists():
    subprocess.run([sys.executable, 'scripts/download_models.py', '--cache_dir', MODEL_CACHE], check=True)
    SENTINEL.touch()
else:
    print('Model cache exists — skipping download.')
cfg.locate_anything['cache_dir'] = MODEL_CACHE

In [ ]:
# Cell P1.2: Inspect model internals (IMPORTANT — run after first download)
# Paste the output in a comment here if the vision encoder path differs from expected.
# This tells you if feature_mode='hidden_state' will work or if you need 'grounding_map'.
from src.locate_anything.worker import LocateAnythingWorker
worker = LocateAnythingWorker(
    cache_dir=MODEL_CACHE, device=DEVICE,
    load_in_4bit=cfg.locate_anything.load_in_4bit,
)
# Loads the model and prints module tree — heavy but cached
worker.load()
worker.inspect_model_internals()
# TODO_VERIFY: confirm vision encoder path matches what is printed above
print('If vision encoder attribute path differs from expected, update _VISION_ENCODER_CANDIDATES in worker.py')

In [ ]:
# Cell P1.3: Region preview on map (verify bounding boxes before GEE export)
# Requires earthengine-api: pip install earthengine-api
import yaml
with open('config/regions_dhaka.yaml') as f:
    regions_cfg = yaml.safe_load(f)

print('Regions loaded. TODO_VERIFY: Open GEE Code Editor, paste gee/01_export_s2_composites.js,')
print('and visually confirm each bounding box before running exports.')
for name, r in regions_cfg['regions'].items():
    bb = r['bbox']
    print(f'  {name}: [{bb["lon_min"]}, {bb["lat_min"]}, {bb["lon_max"]}, {bb["lat_max"]}]')

# Optional: display on folium map if available
try:
    import folium
    m = folium.Map(location=[23.78, 90.41], zoom_start=12)
    colors = ['red', 'blue', 'green', 'orange', 'purple']
    for (name, r), color in zip(regions_cfg['regions'].items(), colors):
        bb = r['bbox']
        folium.Rectangle(
            bounds=[[bb['lat_min'], bb['lon_min']], [bb['lat_max'], bb['lon_max']]],
            color=color, fill=True, fill_opacity=0.2, tooltip=name
        ).add_to(m)
    display(m)
except ImportError:
    print('Install folium for map preview: pip install folium')

In [ ]:
# Cell P1.4: GEE export instructions
print('GEE EXPORT INSTRUCTIONS:')
print('1. Open code.earthengine.google.com')
print('2. Paste and run each script in order:')
for script in ['gee/01_export_s2_composites.js', 'gee/03_weak_labels.js', 'gee/04_export_socioeconomic.js']:
    print(f'   - {script}')
print('3. For GRAM baseline tiles: python gee/02_export_esri_tiles.py --output_dir', cfg.paths.tiles_dir)
print('4. All exports go to Google Drive folder:', cfg.paths.data_dir)
print('5. Wait for GEE Tasks to complete before proceeding.')

In [ ]:
# Cell P1.5: Build dataset manifest (skip if already built)
import json
MANIFEST_PATH = cfg.paths.manifest
if Path(MANIFEST_PATH).exists():
    with open(MANIFEST_PATH) as f:
        manifest = json.load(f)
    print(f'Manifest exists: {manifest["n_tiles"]} tiles. Skipping rebuild.')
else:
    print('Building manifest — run after GEE tiles are in Drive.')
    from src.data.tiles import build_dataset_manifest
    manifest = build_dataset_manifest(
        tiles_dir=cfg.paths.tiles_dir,
        labels_dir=cfg.paths.labels_dir,
        regions_yaml='config/regions_dhaka.yaml',
        output_path=MANIFEST_PATH,
        val_fraction=cfg.data.val_fraction,
        seed=cfg.seed,
    )

## Phase 2 — Weak Labels (one-time, cached)

In [ ]:
# Cell P2.1: Ingest GEE weak-label rasters
from pathlib import Path
from src.data.weak_labels import build_confidence_json
import glob

CONFIDENCE_PATH = cfg.paths.confidence_json
if Path(CONFIDENCE_PATH).exists():
    print('confidence.json exists — skipping geospatial fusion.')
else:
    label_tifs = glob.glob(str(Path(cfg.paths.labels_dir) / 'weeklabels_*.tif'))
    if not label_tifs:
        print('No label TIFs found. Complete GEE export first (Phase 1, Cell P1.4).')
    else:
        build_confidence_json(
            label_tifs=label_tifs,
            output_labels_dir=cfg.paths.labels_dir,
            la_validation_path=cfg.paths.la_validation_json,
            use_la=False,  # LocateAnything validation comes next
            output_confidence_path=CONFIDENCE_PATH,
        )

In [ ]:
# Cell P2.2: LocateAnything zero-shot label validation (Direction B)
# ~30-50 min one-time cost on A100. Cached to la_validation.json.
LA_VAL_PATH = cfg.paths.la_validation_json
if Path(LA_VAL_PATH).exists():
    print('la_validation.json exists — skipping VLM validation.')
elif not cfg.weak_labels.use_locate_anything_validation:
    print('LocateAnything validation disabled (use_locate_anything_validation=false).')
else:
    print('Running LocateAnything label validation (~30-50 min) ...')
    from src.locate_anything.label_validator import run_label_validation
    import json
    with open(cfg.paths.manifest) as f:
        manifest = json.load(f)
    tile_ids = [t['tile_id'] for t in manifest['tiles']]

    def esri_image_loader(tile_id):
        from PIL import Image
        import numpy as np
        # Load ESRI z16 tile for high-res visual validation
        p = Path(cfg.paths.tiles_dir) / f'{tile_id}_esri.png'
        if p.exists():
            return Image.open(str(p)).convert('RGB')
        # Fallback: use S2 RGB tile
        p2 = Path(cfg.paths.tiles_dir) / f'{tile_id}_rgb.tif'
        if p2.exists():
            import rasterio
            with rasterio.open(str(p2)) as src:
                arr = src.read([1, 2, 3]).transpose(1, 2, 0)
            arr = (arr * 255).clip(0, 255).astype('uint8')
            return Image.fromarray(arr)
        raise FileNotFoundError(f'No image for {tile_id}')

    run_label_validation(
        worker=worker,
        tile_ids=tile_ids,
        image_loader=esri_image_loader,
        output_path=LA_VAL_PATH,
        generation_mode=cfg.locate_anything.generation_mode,
        max_new_tokens=cfg.locate_anything.max_new_tokens,
        resume=True,
    )

In [ ]:
# Cell P2.3: Stratify HC vs noisy, render confidence strata figure
from src.viz.plots import fig_confidence_strata
import json

if Path(cfg.paths.confidence_json).exists():
    fig_confidence_strata(
        confidence_json=cfg.paths.confidence_json,
        output_dir=str(Path(cfg.paths.results_dir) / 'figures'),
    )
    with open(cfg.paths.confidence_json) as f:
        conf = json.load(f)
    n_hc = sum(t.get('n_hc', 0) for t in conf.get('tiles', []))
    print(f'Total HC pixels: {n_hc:,}')
else:
    print('confidence.json not found — run Phase 2 Cell P2.1 first.')

## Phase 3 — Feature & SAS-Net Caching (one-time)

In [ ]:
# Cell P3.1: MoonViT feature extraction (all tiles × prompts)
# Most expensive one-time step (~30-60 min). After this, training is fast.
FEAT_DONE_SENTINEL = Path(cfg.paths.features_cache_dir) / 'extraction_done.flag'
if FEAT_DONE_SENTINEL.exists():
    print('Feature cache exists — skipping extraction.')
else:
    print('Extracting MoonViT features (one-time, ~30-60 min) ...')
    from src.locate_anything.feature_extractor import FeatureExtractor
    from src.data.tiles import SlumTileDataset
    import json
    from PIL import Image
    import numpy as np
    import rasterio

    extractor = FeatureExtractor(
        worker=worker,
        cache_dir=cfg.paths.features_cache_dir,
        feature_mode=cfg.locate_anything.feature_mode,
        generation_mode=cfg.locate_anything.generation_mode,
        max_new_tokens=cfg.locate_anything.max_new_tokens,
    )

    with open(cfg.paths.manifest) as f:
        manifest = json.load(f)
    all_tile_ids = [t['tile_id'] for t in manifest['tiles']]

    def rgb_loader(tile_id):
        p = Path(cfg.paths.tiles_dir) / f'{tile_id}_rgb.tif'
        with rasterio.open(str(p)) as src:
            arr = src.read([1, 2, 3]).transpose(1, 2, 0)
        arr = np.clip(arr * 255, 0, 255).astype('uint8')
        return Image.fromarray(arr)

    # Extract for the most general config (full) — covers all prompt variants
    extractor.extract_all_tiles(all_tile_ids, rgb_loader, model_config='full', verbose=True)
    FEAT_DONE_SENTINEL.touch()
    print('Feature extraction complete.')

In [ ]:
# Cell P3.2: Train SAS-Net (once; caches clean tiles)
SASNET_CKPT = Path(cfg.paths.runs_dir) / 'sasnet_best.pt'
CLEAN_SENTINEL = Path(cfg.paths.tiles_dir) / 'clean_tiles_done.flag'

if SASNET_CKPT.exists() and CLEAN_SENTINEL.exists():
    print('SAS-Net checkpoint and clean tiles exist — skipping training.')
else:
    print('Training SAS-Net (~1 A100-hour) ...')
    from src.data.tiles import SlumTileDataset
    from src.data.augment import get_train_transform, get_eval_transform
    from torch.utils.data import DataLoader
    from src.train.train_sasnet import train_sasnet, cache_clean_tiles
    from omegaconf import OmegaConf

    cfg_dict = OmegaConf.to_container(cfg, resolve=True)
    eco_channels = cfg_dict['fusion']['socioeconomic_channels']

    train_ds = SlumTileDataset(
        manifest_path=cfg.paths.manifest, split='train',
        tiles_dir=cfg.paths.tiles_dir, labels_dir=cfg.paths.labels_dir,
        socioeconomic_dir=cfg.paths.socioeconomic_dir,
        socioeconomic_channels=eco_channels,
        transform=get_train_transform(),
    )
    val_ds = SlumTileDataset(
        manifest_path=cfg.paths.manifest, split='val',
        tiles_dir=cfg.paths.tiles_dir, labels_dir=cfg.paths.labels_dir,
        socioeconomic_dir=cfg.paths.socioeconomic_dir,
        socioeconomic_channels=eco_channels,
    )
    train_dl = DataLoader(train_ds, batch_size=cfg_dict['train']['batch_size'],
                          shuffle=True, num_workers=2, pin_memory=True)
    val_dl   = DataLoader(val_ds,   batch_size=cfg_dict['train']['batch_size'],
                          shuffle=False, num_workers=2)

    ckpt_path = train_sasnet(
        config=cfg_dict, train_loader=train_dl, val_loader=val_dl,
        output_dir=cfg.paths.runs_dir, device=DEVICE,
    )

    # Cache clean tiles
    all_dl = DataLoader(train_ds, batch_size=8, shuffle=False, num_workers=2)
    cache_clean_tiles(ckpt_path, all_dl, cfg.paths.tiles_dir, cfg_dict, DEVICE)
    CLEAN_SENTINEL.touch()
    print('SAS-Net training and clean-tile caching complete.')

## Phase 4 — Experiments (cheap, resumable)

In [ ]:
# Cell P4.1: Load experiment matrix and registry
import yaml
from src.tracking.registry import RunRegistry

with open('config/experiments.yaml') as f:
    experiments = yaml.safe_load(f)['experiments']

registry = RunRegistry(cfg.paths.registry)
for exp in experiments:
    registry.register(exp['id'], 'tbd', exp['experiment'])

print(f'Experiment matrix: {len(experiments)} rows')
print(f'Registry status: {registry.summary()}')

FORCE_RERUN = False  # Set True to rerun already-done experiments

In [ ]:
# Cell P4.2: Run experiment orchestration loop
# Trains tiny heads over cached features — each row takes 5-20 min.
# Skips rows already in registry as DONE (resume-safe).
from omegaconf import OmegaConf, DictConfig
from src.train.train_segmenter import train_segmenter
from src.eval.evaluate import evaluate
from src.tracking.recorder import ResultsRecorder
from src.data.tiles import SlumTileDataset
from src.data.augment import get_train_transform, get_eval_transform
from torch.utils.data import DataLoader
import copy

recorder = ResultsRecorder(results_dir=cfg.paths.results_dir)
base_cfg = OmegaConf.to_container(cfg, resolve=True)

for exp in experiments:
    run_id = exp['id']
    if not registry.should_run(run_id, force=FORCE_RERUN):
        print(f'[{run_id}] Already done — skipping.')
        continue

    print(f'\n[{run_id}] Starting: {exp["description"]}')
    registry.set_status(run_id, 'running')

    try:
        # Merge experiment overrides into base config
        run_cfg = copy.deepcopy(base_cfg)
        for key, val in exp.get('overrides', {}).items():
            keys = key.split('.')
            d = run_cfg
            for k in keys[:-1]:
                d = d.setdefault(k, {})
            d[keys[-1]] = val
        run_cfg['_experiment_id'] = exp['experiment']

        eco_channels = run_cfg.get('fusion', {}).get('socioeconomic_channels', [])

        train_ds = SlumTileDataset(
            manifest_path=cfg.paths.manifest, split='train',
            tiles_dir=cfg.paths.tiles_dir, labels_dir=cfg.paths.labels_dir,
            socioeconomic_dir=cfg.paths.socioeconomic_dir,
            socioeconomic_channels=eco_channels,
            transform=get_train_transform(),
        )
        val_ds = SlumTileDataset(
            manifest_path=cfg.paths.manifest, split='val',
            tiles_dir=cfg.paths.tiles_dir, labels_dir=cfg.paths.labels_dir,
            socioeconomic_dir=cfg.paths.socioeconomic_dir,
            socioeconomic_channels=eco_channels,
        )
        test_ds = SlumTileDataset(
            manifest_path=cfg.paths.manifest, split='test',
            tiles_dir=cfg.paths.tiles_dir, labels_dir=cfg.paths.labels_dir,
            socioeconomic_dir=cfg.paths.socioeconomic_dir,
            socioeconomic_channels=eco_channels,
            use_hc_only=True,
        )

        bs = run_cfg.get('train', {}).get('batch_size', 16)
        train_dl = DataLoader(train_ds, batch_size=bs, shuffle=True, num_workers=2, pin_memory=True)
        val_dl   = DataLoader(val_ds,   batch_size=bs, shuffle=False, num_workers=2)
        test_dl  = DataLoader(test_ds,  batch_size=bs, shuffle=False, num_workers=2)

        run_output_dir = str(Path(cfg.paths.runs_dir) / run_id)
        ckpt = train_segmenter(
            config=run_cfg, run_id=run_id,
            train_loader=train_dl, val_loader=val_dl,
            output_dir=run_output_dir, device=DEVICE,
        )

        metrics = evaluate(
            config=run_cfg, run_id=run_id,
            checkpoint_path=ckpt, data_loader=test_dl,
            results_dir=cfg.paths.results_dir, device=DEVICE,
        )

        registry.set_status(run_id, 'done', result_path=run_output_dir)
        print(f'[{run_id}] Done. HC-IoU={metrics.get("hc_iou", "nan"):.4f}')

    except Exception as e:
        registry.set_status(run_id, 'failed')
        print(f'[{run_id}] FAILED: {e}')
        import traceback; traceback.print_exc()

print(f'\nAll experiments done. Registry: {registry.summary()}')

In [ ]:
# Cell P4.3: GRAM head-to-head eval (if enabled)
if base_cfg.get('decisions', {}).get('D6_include_gram_headtohead', True):
    GRAM_DIR = str(PROJECT_ROOT / 'gram_baseline')  # path to existing GRAM outputs
    if Path(GRAM_DIR).exists():
        from src.eval.gram_baseline import evaluate_gram
        import json
        with open(cfg.paths.manifest) as f:
            manifest = json.load(f)
        hc_tile_ids = [t['tile_id'] for t in manifest['tiles'] if t.get('hc_pixel_count', 0) > 0]
        print(f'Running GRAM eval on {len(hc_tile_ids)} HC tiles ...')
        # Labels and HC mask from test dataset would be assembled here
        print('TODO: assemble labels tensor from test dataset and pass to evaluate_gram()')
    else:
        print(f'GRAM output dir not found: {GRAM_DIR}. Skipping head-to-head.')
else:
    print('GRAM head-to-head disabled (D6_include_gram_headtohead=false).')

## Phase 5 — Figures & Tables

In [ ]:
# Cell P5.1: Regenerate all figures and tables
subprocess.run([sys.executable, 'scripts/make_paper_figures.py',
                '--results_dir', cfg.paths.results_dir], check=True)
print('All figures and tables regenerated.')

In [ ]:
# Cell P5.2: Display all figures inline
from IPython.display import Image as IPyImage, display
from pathlib import Path

figures_dir = Path(cfg.paths.results_dir) / 'figures'
for fig_path in sorted(figures_dir.glob('*.png')):
    print(f'\n--- {fig_path.name} ---')
    display(IPyImage(str(fig_path)))

In [ ]:
# Cell P5.3: Print headline results
import pandas as pd
csv_path = str(Path(cfg.paths.results_dir) / 'tables' / 'all_runs.csv')
if Path(csv_path).exists():
    df = pd.read_csv(csv_path)
    cols = ['run_id', 'metric_hc_iou', 'metric_precision', 'metric_recall',
            'metric_f1', 'metric_fpr_control_old_dhaka']
    avail = [c for c in cols if c in df.columns]
    print(df[avail].sort_values('metric_hc_iou', ascending=False).to_string(index=False))
else:
    print('No results yet. Run Phase 4 first.')